In [1]:
import pandas as pd
import os

In [2]:
pd.set_option('display.max_columns', None)

## Data curation: Part I

##### 1. Combine dicomtocsv_series.csv and dicomtocsv_study.csv
##### 2. Group by PatientID

### Read

In [3]:
output_file = "../Data/dicom.xlsx"
dicom = pd.read_excel(output_file)

## Data curation: Part II

##### 1. For both control and cancer, combine files with the same name in different subfolders disregard differnt extension (e.g., .csv, .txt). Control and cancer stored separately

In [4]:
def combine_and_extract(shared_path, study, folder, file, extract_cols):
    """
    Combine files with the same name into a single file, then extract desired columns
    study: Control or Cancer
    folder: folders under Control or Cancer
    file: which files to process
    extract_cols (list): extracted columns based on file_param 
    """
    count = 0
    for f in folder:
        if count == 0:
            try:
                # for txt #
                df = pd.read_csv(os.path.join(shared_path, study, f, file) + ".txt", sep='|')
            except:
                # for csv #
                df = pd.read_csv(os.path.join(shared_path, study, f, file) + ".csv", encoding='latin-1')
            
        else:
            try:
                # for txt #
                tmp = pd.read_csv(os.path.join(shared_path, study, f, file) + ".txt", sep='|')
            except:
                # for csv #
                try:
                    tmp = pd.read_csv(os.path.join(shared_path, study, f, file) + ".csv", encoding='latin-1')
                except: 
                    print("N/A")
                
                
            try:
                df = pd.concat([df, tmp])
            except: 
                print("No file to concat: file not exist in " + os.path.join(shared_path, study, f))
        
        count += 1
        

#     # Remove duplicated entries
#     df_Xdup = df.drop_duplicates(subset=df.columns.tolist(), keep = 'first').reset_index(drop = True)
    # Sort by 'PATIENT_STUDY_ID
    df.sort_values(by='PATIENT_STUDY_ID', inplace=True, ignore_index=True)
        
    return df[extract_cols]

### Control/Cancer

<span style="color: red;">Choose **Control or Cancer**, change study, folder, and study_ext</span>

In [5]:
shared_path = "../Data/Lee, Ju Hun's files - R3Data"

In [6]:
# study = "Control"
# folder = ["R3_3787_Lee_Control_Extract_Files", "R3_3787_Lee_Data_Controls_20240508"]
# study_ext = "_controls"

study = "Cancer"
folder = ["R3_3787_Lee_Cancer_Extract_Files", "R3_3787_Lee_Data_Cancer_20240509", "R3_3787_Lee_Data_Cancer_20250912"]
study_ext = ""

### Files

See **parameter explanation for detail information

In [7]:
file_path = "../Data/parameters of interest.xlsx"
files = pd.ExcelFile(file_path).sheet_names

In [8]:
files

['pathology',
 'pathology_findings',
 'patient_data_ie',
 'family_hx',
 'patient_demo',
 'vitals',
 'risk_factors',
 'enteredit_findings',
 'hormonal_mens']

#### 1. pathology

<span style="color: blue;">Change idx value **based on file**</span>

In [59]:
idx = 1
file = files[idx-1]
print(file)
extract_cols = pd.read_excel(file_path, sheet_name=file)["Name"].tolist()
print(extract_cols)

pathology
['PATIENT_STUDY_ID', 'BX_ID', 'PATHOLOGY_DATE', 'LESION_CLASS', 'SIDE']


In [60]:
df = combine_and_extract(shared_path, study, folder, file + study_ext, extract_cols)

N/A
No file to concat: file not exist in ../Data/Lee, Ju Hun's files - R3Data/Cancer/R3_3787_Lee_Data_Cancer_20240509


<span style="color: blue;">Reformat PATHOLOGY_DATE **based on file**</span>

In [61]:
df['PATHOLOGY_DATE'] = pd.to_datetime(df['PATHOLOGY_DATE']).dt.strftime('%Y-%m-%d')

<span style="color: cyan;">Duplicated entries **based on file**</span>

In [62]:
df[df.duplicated()]

,PATIENT_STUDY_ID,BX_ID,PATHOLOGY_DATE,LESION_CLASS,SIDE
10,4330103113,425798,2018-01-29,Benign,R
23,4330214688,478250,2021-04-06,Benign,R
26,4330251952,462069,2021-12-15,Benign,R
32,4330268555,429934,2018-09-28,Benign,L
48,4330331505,422139,2018-05-09,Malignant,R
...,...,...,...,...,...
14030,4339566815,418774,2019-09-26,Benign,L
14048,4339601532,423569,2018-03-12,Benign,R
14059,4339778947,466925,2022-05-25,Benign,L
14066,4339933706,441680,2017-07-10,Benign,L


<span style="color: blue;">Remove duplicated entries **based on file**</span>

In [63]:
df_Xdup = df.drop_duplicates(subset=df.columns.tolist(), keep = 'first').reset_index(drop = True)

<span style="color: blue;">Change sort_values **based on file**</span>

In [64]:
df_Xdup.sort_values(by=['PATIENT_STUDY_ID', 'PATHOLOGY_DATE', 'BX_ID'], ascending=[True, False, False], inplace=True, ignore_index=True)

In [65]:
output_file = os.path.join("../Data/", study, "Cleaned", file + ".xlsx")
df_Xdup.to_excel(output_file, index=False)

<ipython-input-65-0e7ad8d14de2>:2: UserWarning: Pandas requires version '1.4.3' or newer of 'xlsxwriter' (version '1.3.7' currently installed).
  df_Xdup.to_excel(output_file, index=False)


#### 2. pathology_findings

<span style="color: blue;">Change idx value **based on file**</span>

In [89]:
idx = 2
file = files[idx-1]
print(file)
extract_cols = pd.read_excel(file_path, sheet_name=file)["Name"].tolist()
print(extract_cols)

pathology_findings
['PATIENT_STUDY_ID', 'BX_ID', 'PATHOLOGY_DATE', 'LESION_CLASS', 'PATHOLOGY_CD', 'HISTROLOGY_GRADE', 'ESTROGEN_RECEPTOR', 'PROGESTERONE_RECEPTOR', 'HER2NEU', 'STAGE_T', 'STAGE_N', 'STAGE_M', 'STAGE_NUM', 'MARGIN_STATUS']


In [90]:
df = combine_and_extract(shared_path, study, folder, file + study_ext, extract_cols)

<span style="color: blue;">Reformat PATHOLOGY_DATE **based on file**</span>

In [91]:
df['PATHOLOGY_DATE'] = pd.to_datetime(df['PATHOLOGY_DATE']).dt.strftime('%Y-%m-%d')

<span style="color: cyan;">Duplicated entries **based on file**</span>

In [92]:
df[df.duplicated()]

,PATIENT_STUDY_ID,BX_ID,PATHOLOGY_DATE,LESION_CLASS,PATHOLOGY_CD,HISTROLOGY_GRADE,ESTROGEN_RECEPTOR,PROGESTERONE_RECEPTOR,HER2NEU,STAGE_T,STAGE_N,STAGE_M,STAGE_NUM,MARGIN_STATUS
28,4333008024,410317,2019-06-17,BENIGN,AD,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
56,4333062366,475428,2020-08-06,BENIGN,FC,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
57,4333062366,471969,2020-12-04,BENIGN,PA,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
83,4333101704,355101,2022-07-15,BENIGN,BC,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
151,4333373719,471839,2020-10-29,BENIGN,BC,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
152,4333373719,471830,2020-10-29,BENIGN,BC,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
168,4333488305,417825,2019-11-07,BENIGN,BC,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
173,4333514173,478396,2021-03-19,BENIGN,AN,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
177,4333571001,427540,2018-11-12,BENIGN,BC,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
191,4333624870,469223,2022-02-21,BENIGN,FC,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


<span style="color: blue;">Remove duplicated entries **based on file**</span>

In [94]:
df_Xdup = df.drop_duplicates(subset=df.columns.tolist(), keep = 'first').reset_index(drop = True)

<span style="color: blue;">Change sort_values **based on file**</span>

In [95]:
df_Xdup.sort_values(by=['PATIENT_STUDY_ID', 'PATHOLOGY_DATE', 'BX_ID'], ascending=[True, False, False], inplace=True, ignore_index=True)

In [96]:
output_file = os.path.join("../Data/", study, "Cleaned", file + ".xlsx")
df_Xdup.to_excel(output_file, index=False)

<ipython-input-96-0e7ad8d14de2>:2: UserWarning: Pandas requires version '1.4.3' or newer of 'xlsxwriter' (version '1.3.7' currently installed).
  df_Xdup.to_excel(output_file, index=False)


#### 3. patient_data_ie

<span style="color: blue;">Change idx value **based on file**</span>

In [9]:
idx = 3
file = files[idx-1]
print(file)
extract_cols = pd.read_excel(file_path, sheet_name=file)["Name"].tolist()
print(extract_cols)

patient_data_ie
['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'ORG_NAME', 'EXAM_CODE', 'EXAM_NAME', 'COMPLETED_DATE', 'ORDER_DATE', 'EXAM_TYPE', 'FIRST_MAMMO_IND', 'LAST_ACTIVITY_DATE', 'INTERNAL_EXAM_ID', 'PATIENT_ID']


In [10]:
df = combine_and_extract(shared_path, study, folder, file + study_ext, extract_cols)

/var/folders/g9/k5r102q54f126nrlr692zy6c0000gn/T/ipykernel_30628/3601793897.py:26: DtypeWarning: Columns (22) have mixed types. Specify dtype option on import or set low_memory=False.
  tmp = pd.read_csv(os.path.join(shared_path, study, f, file) + ".csv", encoding='latin-1')


In [11]:
df

,PATIENT_STUDY_ID,ACCESSION_NUMBER,ORG_NAME,EXAM_CODE,EXAM_NAME,COMPLETED_DATE,ORDER_DATE,EXAM_TYPE,FIRST_MAMMO_IND,LAST_ACTIVITY_DATE,INTERNAL_EXAM_ID,PATIENT_ID
0,4330018595,69461777,Magee Womens Imaging Oakland,MRBRSTBIWX,MR BREAST BILATERAL WWO CONTRAST WWO CAD,06/30/2020,06/23/2020,Breast MRI,N,01/05/2021,83657913,9654707
1,4330018595,63027507,Magee Womens Imaging North,BISCRDBITO,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,07/26/2019,07/08/2019,Mammography,Y,01/05/2021,90728533,9654707
2,4330018595,66064198,Magee Womens Imaging North,BIDIADBITO,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,06/08/2021,04/26/2021,Mammography,N,10/19/2021,89243522,9654707
3,4330018595,69058456,Magee Womens Imaging Oakland,BISPECIMEN,BREAST SURGICAL SPECIMEN,07/29/2020,NaN,Specimen Imaging,N,01/05/2021,82349369,9654707
4,4330018595,69089910,Magee Womens Imaging Oakland,BURSLRTEA,ULTRASOUND GUIDED RADIOACTIVE SEED LOC RIGHT E...,07/28/2020,NaN,Interventional procedure,N,01/05/2021,82371743,9654707
...,...,...,...,...,...,...,...,...,...,...,...,...
161814,4339959661,68434457,UPMC Hamot Imaging Center,BISCRDBITO,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,11/09/2020,09/14/2020,Mammography,N,12/22/2020,82737072,7269813
161815,4339959661,454148687,UPMC Hamot Imaging Center,BISCRDBITO,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,11/10/2021,09/30/2021,Mammography,N,11/22/2021,87437824,7269813
161816,4339959661,453550820,UPMC Hamot Breast Imaging,BIDIADGPPL,DIAG MAMMO PROD DIRECT DIGITAL IMAGES LT POST ...,11/22/2021,NaN,Interventional procedure,N,11/22/2021,87859991,7269813
161817,4339959661,453550821,UPMC Hamot Breast Imaging,BIBXBVACLT,BREAST STEREO BX VAC ASSIST LT WWO SPEC IMG,11/22/2021,NaN,Digital Mammography,N,11/22/2021,87859992,7269813


<span style="color: blue;">Reformat CONTACT_DATE **based on file**</span>

In [12]:
df['COMPLETED_DATE'] = pd.to_datetime(df['COMPLETED_DATE']).dt.strftime('%Y-%m-%d')

<span style="color: blue;">Reformat ORDER_DATE **based on file**</span>

In [13]:
df['ORDER_DATE'] = pd.to_datetime(df['ORDER_DATE']).dt.strftime('%Y-%m-%d')

<span style="color: blue;">Remove duplicated entries **based on file**</span>

In [14]:
df_Xdup = df.drop_duplicates(subset=df.columns.tolist(), keep = 'first').reset_index(drop = True)

In [15]:
df_Xdup

,PATIENT_STUDY_ID,ACCESSION_NUMBER,ORG_NAME,EXAM_CODE,EXAM_NAME,COMPLETED_DATE,ORDER_DATE,EXAM_TYPE,FIRST_MAMMO_IND,LAST_ACTIVITY_DATE,INTERNAL_EXAM_ID,PATIENT_ID
0,4330018595,69461777,Magee Womens Imaging Oakland,MRBRSTBIWX,MR BREAST BILATERAL WWO CONTRAST WWO CAD,2020-06-30,2020-06-23,Breast MRI,N,01/05/2021,83657913,9654707
1,4330018595,63027507,Magee Womens Imaging North,BISCRDBITO,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,2019-07-26,2019-07-08,Mammography,Y,01/05/2021,90728533,9654707
2,4330018595,66064198,Magee Womens Imaging North,BIDIADBITO,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,2021-06-08,2021-04-26,Mammography,N,10/19/2021,89243522,9654707
3,4330018595,69058456,Magee Womens Imaging Oakland,BISPECIMEN,BREAST SURGICAL SPECIMEN,2020-07-29,NaN,Specimen Imaging,N,01/05/2021,82349369,9654707
4,4330018595,69089910,Magee Womens Imaging Oakland,BURSLRTEA,ULTRASOUND GUIDED RADIOACTIVE SEED LOC RIGHT E...,2020-07-28,NaN,Interventional procedure,N,01/05/2021,82371743,9654707
...,...,...,...,...,...,...,...,...,...,...,...,...
120283,4339959661,68434457,UPMC Hamot Imaging Center,BISCRDBITO,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,2020-11-09,2020-09-14,Mammography,N,12/22/2020,82737072,7269813
120284,4339959661,454148687,UPMC Hamot Imaging Center,BISCRDBITO,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,2021-11-10,2021-09-30,Mammography,N,11/22/2021,87437824,7269813
120285,4339959661,453550820,UPMC Hamot Breast Imaging,BIDIADGPPL,DIAG MAMMO PROD DIRECT DIGITAL IMAGES LT POST ...,2021-11-22,NaN,Interventional procedure,N,11/22/2021,87859991,7269813
120286,4339959661,453550821,UPMC Hamot Breast Imaging,BIBXBVACLT,BREAST STEREO BX VAC ASSIST LT WWO SPEC IMG,2021-11-22,NaN,Digital Mammography,N,11/22/2021,87859992,7269813


<span style="color: blue;">Change sort_values **based on file**</span>

In [16]:
df_Xdup.sort_values(by=['PATIENT_STUDY_ID', 'COMPLETED_DATE'], ascending=[True, True], inplace=True, ignore_index=True)

In [17]:
output_file = os.path.join("../Data/", study, "Cleaned", file + ".xlsx")
df_Xdup.to_excel(output_file, index=False)

#### 4. family_hx

<span style="color: blue;">Change idx value **based on file**</span>

In [25]:
idx = 4
file = files[idx-1]
print(file)
extract_cols = pd.read_excel(file_path, sheet_name=file)["Name"].tolist()
print(extract_cols)

family_hx
['PATIENT_STUDY_ID', 'LINE_NUM', 'CONTACT_DATE', 'MEDICAL_HX_TITLE', 'RELATION_TITLE']


In [98]:
df = combine_and_extract(shared_path, study, folder, file + study_ext, extract_cols)

<span style="color: blue;">Reformat CONTACT_DATE **based on file**</span>

In [99]:
df['CONTACT_DATE'] = pd.to_datetime(df['CONTACT_DATE']).dt.strftime('%Y-%m-%d')

<span style="color: cyan;">Duplicated entries **based on file**</span>

In [100]:
df[df.duplicated()]

,PATIENT_STUDY_ID,LINE_NUM,CONTACT_DATE,MEDICAL_HX_TITLE,RELATION_TITLE
526,4330000534,4,2018-10-25,ARTHRITIS,BIOLOGICAL MOTHER
527,4330000534,3,2018-10-25,RETINAL DETACHMENT,BIOLOGICAL MOTHER
528,4330000534,2,2018-10-25,HYPERTENSION,BIOLOGICAL MOTHER
529,4330000534,1,2018-10-25,MIGRAINES,SISTER
530,4330000534,9,2018-10-25,ARTHRITIS,OTHER
...,...,...,...,...,...
12570920,4339949064,4,2019-07-15,DIABETES,SISTER
12570922,4339949064,1,2019-07-15,DIABETES,BIOLOGICAL MOTHER
12570923,4339949064,3,2022-08-15,"CA, LIVER",BIOLOGICAL FATHER
12570924,4339949064,3,2019-12-03,"CA, LIVER",BIOLOGICAL FATHER


In [101]:
df[(df["PATIENT_STUDY_ID"] == 4330114580) & (df["LINE_NUM"] == 7)].sort_values(by='CONTACT_DATE')

,PATIENT_STUDY_ID,LINE_NUM,CONTACT_DATE,MEDICAL_HX_TITLE,RELATION_TITLE


<span style="color: blue;">Remove duplicated entries **based on file**</span>

In [102]:
df_Xdup = df.drop_duplicates(subset=df.columns.tolist(), keep = 'first').reset_index(drop = True)

<span style="color: blue;">Change sort_values **based on file**</span>

In [103]:
df_Xdup.sort_values(by=['PATIENT_STUDY_ID', 'CONTACT_DATE'], ascending=[True, False], inplace=True, ignore_index=True)

In [104]:
output_file = os.path.join("../Data/", study, "Cleaned", file + ".txt")
df_Xdup.to_csv(output_file, header=True, index=None, sep=' ')

#### 5. patient_demo

<span style="color: blue;">Change idx value **based on file**</span>

In [26]:
idx = 5
file = files[idx-1]
print(file)
extract_cols = pd.read_excel(file_path, sheet_name=file)["Name"].tolist()
print(extract_cols)

patient_demo
['PATIENT_STUDY_ID', 'BIRTH_DATE', 'GENDER_TITLE', 'RACE_TITLE', 'ETHNIC_TITLE']


In [107]:
df = combine_and_extract(shared_path, study, folder, file + study_ext, extract_cols)

<span style="color: blue;">Reformat BIRTH_DATE **based on file**</span>

In [108]:
df['BIRTH_DATE'] = pd.to_datetime(df['BIRTH_DATE']).dt.strftime('%Y-%m-%d')

<span style="color: cyan;">Duplicated entries **based on file**</span>

In [109]:
df[df.duplicated()]

,PATIENT_STUDY_ID,BIRTH_DATE,GENDER_TITLE,RACE_TITLE,ETHNIC_TITLE
33,4330115804,1973-07-01,FEMALE,WHITE,NaN
92,4330242158,1970-07-01,FEMALE,WHITE,NaN
153,4330312676,1973-07-01,FEMALE,WHITE,NOT SPECIFIED
171,4330318623,1960-07-01,FEMALE,WHITE,NaN
178,4330320873,1962-07-01,FEMALE,NOT SPECIFIED,NaN
...,...,...,...,...,...
49155,4337981130,1975-07-01,FEMALE,WHITE,NaN
49192,4339042307,1953-07-01,FEMALE,WHITE,NaN
49283,4339515366,1970-07-01,FEMALE,WHITE,NaN
49371,4339771691,1966-07-01,FEMALE,WHITE,NOT SPECIFIED


In [110]:
df[df["PATIENT_STUDY_ID"] == 4330170072]

,PATIENT_STUDY_ID,BIRTH_DATE,GENDER_TITLE,RACE_TITLE,ETHNIC_TITLE


<span style="color: blue;">Remove duplicated entries **based on file**</span>

In [111]:
df_Xdup = df.drop_duplicates(subset=df.columns.tolist(), keep = 'first').reset_index(drop = True)

<span style="color: blue;">Change sort_values **based on file**</span>

In [112]:
df_Xdup.sort_values(by=['PATIENT_STUDY_ID', 'BIRTH_DATE'], ascending=[True, False], inplace=True, ignore_index=True)

In [113]:
output_file = os.path.join("../Data/", study, "Cleaned", file + ".xlsx")
df_Xdup.to_excel(output_file, index=False)

<ipython-input-113-0e7ad8d14de2>:2: UserWarning: Pandas requires version '1.4.3' or newer of 'xlsxwriter' (version '1.3.7' currently installed).
  df_Xdup.to_excel(output_file, index=False)


#### 6. vitals

<span style="color: blue;">Change idx value **based on file**</span>

In [27]:
idx = 6
file = files[idx-1]
print(file)
extract_cols = pd.read_excel(file_path, sheet_name=file)["Name"].tolist()
print(extract_cols)

vitals
['PATIENT_STUDY_ID', 'DATE_TAKEN', 'WEIGHT', 'WEIGHT_UNIT', 'HEIGHT', 'HEIGHT_UNIT', 'BMI']


In [115]:
df = combine_and_extract(shared_path, study, folder, file + study_ext, extract_cols)

<span style="color: blue;">Reformat date **based on file**</span>

In [116]:
df['DATE_TAKEN'] = pd.to_datetime(df['DATE_TAKEN'], format='%m/%d/%Y %H:%M:%S')
df['DATE_TAKEN'] = df['DATE_TAKEN'].dt.strftime('%Y-%m-%d')

<span style="color: cyan;">Duplicated entries **based on file**</span>

In [117]:
df[df.duplicated()]

,PATIENT_STUDY_ID,DATE_TAKEN,WEIGHT,WEIGHT_UNIT,HEIGHT,HEIGHT_UNIT,BMI
25,4330000534,2018-10-06,68.0,KG,NaN,CM,25.00
29,4330000534,2018-10-05,NaN,KG,NaN,CM,25.00
113,4330082402,2017-10-26,2624.0,OZ,67.0,IN,25.69
169,4330109886,2017-10-10,2096.0,OZ,64.0,IN,22.49
172,4330109886,2022-07-04,2160.0,OZ,64.0,IN,23.17
...,...,...,...,...,...,...,...
1137415,4339949064,2022-11-18,1760.0,OZ,61.0,IN,20.78
1137416,4339949064,2020-06-24,1776.0,OZ,61.0,IN,20.97
1137418,4339949064,2019-07-15,1808.0,OZ,61.0,IN,21.35
1137421,4339949064,2022-08-15,1731.2,OZ,61.0,IN,20.44


In [118]:
df[df["PATIENT_STUDY_ID"] == 4330103113].sort_values(by='DATE_TAKEN')

,PATIENT_STUDY_ID,DATE_TAKEN,WEIGHT,WEIGHT_UNIT,HEIGHT,HEIGHT_UNIT,BMI


<span style="color: blue;">Remove duplicated entries **based on file**</span>

In [119]:
df_Xdup = df.drop_duplicates(subset=df.columns.tolist(), keep = 'first').reset_index(drop = True)

<span style="color: blue;">Change sort_values **based on file**</span>

In [120]:
df_Xdup.sort_values(by=['PATIENT_STUDY_ID', 'DATE_TAKEN'], ascending=[True, False], inplace=True, ignore_index=True)

In [121]:
df_Xdup

,PATIENT_STUDY_ID,DATE_TAKEN,WEIGHT,WEIGHT_UNIT,HEIGHT,HEIGHT_UNIT,BMI
0,4330000534,2022-04-19,2848.0,OZ,65.0,IN,29.62
1,4330000534,2021-06-29,2976.0,OZ,65.0,IN,30.95
2,4330000534,2021-03-02,2896.0,OZ,65.0,IN,30.12
3,4330000534,2020-08-27,2944.0,OZ,65.0,IN,30.62
4,4330000534,2020-03-02,2560.0,OZ,66.0,IN,25.82
...,...,...,...,...,...,...,...
859060,4339982047,2021-04-15,2864.0,OZ,64.0,IN,30.73
859061,4339982047,2018-10-17,2944.0,OZ,64.0,IN,31.58
859062,4339988199,2022-05-26,3072.0,OZ,61.0,IN,36.28
859063,4339988199,2022-04-21,3216.0,OZ,61.0,IN,37.98


<span style="color: blue;">Identify entry without BMI OR lack of either weight or height to calculate BMI</span>

In [122]:
df_Xbmi = df_Xdup[df_Xdup["BMI"].isna()]
df_Xbmi2 = df_Xbmi[df_Xbmi["WEIGHT"].isna() | df_Xbmi["HEIGHT"].isna()]
df_Xbmi_idx = df_Xbmi.index
df_Xbmi2_idx = df_Xbmi2.index

In [123]:
df_Xbmi_idx.equals(df_Xbmi2_idx)

True

In [124]:
df_Xbmi_idx

Index([    21,    548,    549,    878,   1173,   1376,   1377,   1378,   1379,
         2099,
       ...
       857804, 857854, 858036, 858479, 858482, 858579, 858746, 858929, 858930,
       859051],
      dtype='int64', length=3288)

<span style="color: blue;">Remove entries without BMI</span>

In [125]:
df_Xdup.drop(df_Xbmi_idx, inplace=True)
df_Xdup.drop(['WEIGHT', 'WEIGHT_UNIT', 'HEIGHT', 'HEIGHT_UNIT'], axis=1, inplace=True)

In [126]:
output_file = os.path.join("../Data/", study, "Cleaned", file + ".xlsx")
df_Xdup.to_excel(output_file, index=False)

<ipython-input-126-0e7ad8d14de2>:2: UserWarning: Pandas requires version '1.4.3' or newer of 'xlsxwriter' (version '1.3.7' currently installed).
  df_Xdup.to_excel(output_file, index=False)


#### 7. risk_factors

<span style="color: blue;">Change idx value **based on file**</span>

In [28]:
idx = 7
file = files[idx-1]
print(file)
extract_cols = pd.read_excel(file_path, sheet_name=file)["Name"].tolist()
print(extract_cols)

risk_factors
['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'RISK_FACTOR_CD', 'RISK_FACTOR_NAME', 'RISK_SEQUENCE', 'EXAM_COMPLETED_DATE']


In [67]:
df = combine_and_extract(shared_path, study, folder, file + study_ext, extract_cols)

<span style="color: blue;">Standardize date **based on file**</span>

In [68]:
df['EXAM_COMPLETED_DATE'] = pd.to_datetime(df['EXAM_COMPLETED_DATE'], format='%m/%d/%Y')
df['EXAM_COMPLETED_DATE'] = df['EXAM_COMPLETED_DATE'].dt.strftime('%Y-%m-%d')

<span style="color: cyan;">Duplicated entries **based on file**</span>

In [69]:
df[df.duplicated()]

,PATIENT_STUDY_ID,ACCESSION_NUMBER,RISK_FACTOR_CD,RISK_FACTOR_NAME,RISK_SEQUENCE,EXAM_COMPLETED_DATE
73,4330103113,78547611,OM,"Family history of ovarian cancer in mother, si...",19,2018-01-29
74,4330103113,78547611,3,Very strong family history of breast cancer (m...,16,2018-01-29
78,4330103113,78547611,3,Very strong family history of breast cancer (m...,16,2018-01-29
79,4330103113,78547611,OM,"Family history of ovarian cancer in mother, si...",19,2018-01-29
80,4330103113,61304464,3,Very strong family history of breast cancer (m...,16,2020-01-18
...,...,...,...,...,...,...
512011,4339933706,71871349,Q,Post-menopausal patient,2,2017-07-10
512012,4339933706,454877701,Q,Post-menopausal patient,2,2021-11-17
512013,4339933706,70511846,1,"Weak family history of breast cancer (aunt, gr...",14,2017-07-25
512014,4339933706,70511801,1,"Weak family history of breast cancer (aunt, gr...",14,2017-07-28


<span style="color: blue;">Remove duplicated entries **based on file**</span>

In [70]:
df_Xdup = df.drop_duplicates(subset=df.columns.tolist(), keep = 'first').reset_index(drop = True)

<span style="color: blue;">Change sort_values **based on file**</span>

In [71]:
df_Xdup.sort_values(by=['PATIENT_STUDY_ID', 'EXAM_COMPLETED_DATE', 'ACCESSION_NUMBER'], ascending=[True, False, False], inplace=True, ignore_index=True)

In [72]:
output_file = os.path.join("../Data/", study, "Cleaned", file + ".xlsx")
df_Xdup.to_excel(output_file, index=False)

<ipython-input-72-0e7ad8d14de2>:2: UserWarning: Pandas requires version '1.4.3' or newer of 'xlsxwriter' (version '1.3.7' currently installed).
  df_Xdup.to_excel(output_file, index=False)


#### 9. hormonal_mens

<span style="color: blue;">Change idx value **based on file**</span>

In [29]:
idx = 9
file = files[idx-1]
print(file)
extract_cols = pd.read_excel(file_path, sheet_name=file)["Name"].tolist()
print(extract_cols)

hormonal_mens
['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'AGE_FIRST_USE', 'AGE_LAST_USE', 'DURATION', 'CURRENT_USE_IND', 'NEVER_USE_IND', 'AGE_MENARCHE', 'AGE_FIRST_LIVE_BIRTH', 'AGE_MENOPAUSE', 'AGE_HYSTERECTOMY', 'AGE_RIGHT_OVARY_REMOVAL', 'AGE_LEFT_OVARY_REMOVAL', 'PARITY_COUNT', 'PREGNANCY_COUNT', 'LAST_MENSTRUAL_DATE', 'MENSTRUAL_STATUS_CD', 'EXAM_COMPLETED_DATE']


In [92]:
df = combine_and_extract(shared_path, study, folder, file + study_ext, extract_cols)

<span style="color: blue;">Standardize date **based on file**</span>

In [93]:
df['EXAM_COMPLETED_DATE'] = pd.to_datetime(df['EXAM_COMPLETED_DATE'], format='%m/%d/%Y')
df['EXAM_COMPLETED_DATE'] = df['EXAM_COMPLETED_DATE'].dt.strftime('%Y-%m-%d')

<span style="color: cyan;">Duplicated entries **based on file**</span>

In [94]:
df[df.duplicated()]

,PATIENT_STUDY_ID,ACCESSION_NUMBER,AGE_FIRST_USE,AGE_LAST_USE,DURATION,CURRENT_USE_IND,NEVER_USE_IND,AGE_MENARCHE,AGE_FIRST_LIVE_BIRTH,AGE_MENOPAUSE,AGE_HYSTERECTOMY,AGE_RIGHT_OVARY_REMOVAL,AGE_LEFT_OVARY_REMOVAL,PARITY_COUNT,PREGNANCY_COUNT,LAST_MENSTRUAL_DATE,MENSTRUAL_STATUS_CD,EXAM_COMPLETED_DATE
1,4330018595,60695725,0,0,0,N,Y,16,31,49,0,0,0,2.0,3.0,NaN,NaN,2020-06-02
3,4330018595,60103700,0,0,0,N,Y,16,31,49,0,0,0,2.0,3.0,NaN,NaN,2020-06-02
5,4330018595,60103700,0,0,0,N,Y,16,31,49,0,0,0,2.0,3.0,NaN,NaN,2020-06-02
6,4330018595,60103700,0,0,0,N,Y,16,31,49,0,0,0,2.0,3.0,NaN,NaN,2020-06-02
7,4330018595,66064198,0,0,0,N,Y,16,30,49,0,0,0,2.0,3.0,NaN,POSTNAT,2021-06-08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1273015,4339959661,62216536,0,0,0,N,Y,13,0,0,0,0,0,0.0,0.0,11/05/2020 00:00:00,PRE,2019-10-28
1273016,4339959661,453550821,0,0,0,N,Y,13,0,0,0,0,0,0.0,0.0,10/28/2021 00:00:00,PRE,2021-11-22
1273017,4339959661,453550821,0,0,0,N,Y,13,0,0,0,0,0,0.0,0.0,10/28/2021 00:00:00,PRE,2021-11-22
1273018,4339959661,454612462,0,0,0,N,Y,13,0,0,0,0,0,0.0,0.0,10/28/2021 00:00:00,PRE,2021-11-15


In [95]:
df[(df["PATIENT_STUDY_ID"] == 4330103113) & (df["ACCESSION_NUMBER"] == 79000052)].sort_values(by='EXAM_COMPLETED_DATE')

,PATIENT_STUDY_ID,ACCESSION_NUMBER,AGE_FIRST_USE,AGE_LAST_USE,DURATION,CURRENT_USE_IND,NEVER_USE_IND,AGE_MENARCHE,AGE_FIRST_LIVE_BIRTH,AGE_MENOPAUSE,AGE_HYSTERECTOMY,AGE_RIGHT_OVARY_REMOVAL,AGE_LEFT_OVARY_REMOVAL,PARITY_COUNT,PREGNANCY_COUNT,LAST_MENSTRUAL_DATE,MENSTRUAL_STATUS_CD,EXAM_COMPLETED_DATE
355,4330103113,79000052,25,26,0,N,N,12,32,0,39,0,0,2.0,6.0,NaN,POSTSUR,2018-01-09
356,4330103113,79000052,0,0,0,N,Y,12,32,0,39,0,0,2.0,6.0,NaN,POSTSUR,2018-01-09
357,4330103113,79000052,0,0,0,N,Y,12,32,0,39,0,0,2.0,6.0,NaN,POSTSUR,2018-01-09
360,4330103113,79000052,0,0,0,N,Y,12,32,0,39,0,0,2.0,6.0,NaN,POSTSUR,2018-01-09
361,4330103113,79000052,0,0,0,N,Y,12,32,0,39,0,0,2.0,6.0,NaN,POSTSUR,2018-01-09
365,4330103113,79000052,0,0,0,N,Y,12,32,0,39,0,0,2.0,6.0,NaN,POSTSUR,2018-01-09
393,4330103113,79000052,0,0,0,N,Y,12,32,0,39,0,0,2.0,6.0,NaN,POSTSUR,2018-01-09
403,4330103113,79000052,0,0,0,N,Y,12,32,0,39,0,0,2.0,6.0,NaN,POSTSUR,2018-01-09
404,4330103113,79000052,0,0,0,N,Y,12,32,0,39,0,0,2.0,6.0,NaN,POSTSUR,2018-01-09
405,4330103113,79000052,25,26,0,N,N,12,32,0,39,0,0,2.0,6.0,NaN,POSTSUR,2018-01-09


<span style="color: blue;">Remove duplicated entries **based on file**</span>

In [96]:
df_Xdup = df.drop_duplicates(subset=df.columns.tolist(), keep = 'first').reset_index(drop = True)

<span style="color: blue;">Change sort_values **based on file**</span>

In [97]:
df_Xdup.sort_values(by=['PATIENT_STUDY_ID', 'EXAM_COMPLETED_DATE', 'ACCESSION_NUMBER'], ascending=[True, False, False], inplace=True, ignore_index=True)

In [98]:
output_file = os.path.join("../Data/", study, "Cleaned", file + ".xlsx")
df_Xdup.to_excel(output_file, index=False)

<ipython-input-98-0e7ad8d14de2>:2: UserWarning: Pandas requires version '1.4.3' or newer of 'xlsxwriter' (version '1.3.7' currently installed).
  df_Xdup.to_excel(output_file, index=False)


### <span style="color:#FF6347;">**READ**</span> file

In [30]:
study = "Cancer"
file = "risk_factors"

In [31]:
output_file = os.path.join("../Data/", study, "Cleaned", file + ".xlsx")
df = pd.read_excel(output_file)

In [32]:
df

,PATIENT_STUDY_ID,ACCESSION_NUMBER,RISK_FACTOR_CD,RISK_FACTOR_NAME,RISK_SEQUENCE,EXAM_COMPLETED_DATE
0,4330018595,452973800,0,No family history of breast cancer,13,2022-06-09
1,4330018595,452973800,C,Personal breast cancer history,3,2022-06-09
2,4330018595,66064198,C,Personal breast cancer history,3,2021-06-08
3,4330018595,66064198,0,No family history of breast cancer,13,2021-06-08
4,4330018595,69058456,C,Personal breast cancer history,3,2020-07-29
...,...,...,...,...,...,...
337136,4339959661,68434457,1,"Weak family history of breast cancer (aunt, gr...",14,2020-11-09
337137,4339959661,62216536,1,"Weak family history of breast cancer (aunt, gr...",14,2019-10-28
337138,4339959661,62216536,U,Nulliparous,4,2019-10-28
337139,4339959661,78525169,1,"Weak family history of breast cancer (aunt, gr...",14,2018-04-16


----

### Listing out all possible parameters in respective file (Study = Cancer)

In [12]:
file_path = "../Data/Lee, Ju Hun's files - R3Data/Cancer/R3_3787_Lee_Data_Cancer_20240509"

In [13]:
files = [f for f in os.listdir(file_path) if os.path.isfile(os.path.join(file_path, f))]

In [18]:
f = files[0][:-4]
df_csv = pd.read_csv(os.path.join(file_path, files[0]), encoding='latin-1')
df = pd.DataFrame({'Name': df_csv.columns.tolist()})
df.to_excel("../Data/parameters.xlsx", sheet_name=f, index=False)

N = len(files)
i = 2
with pd.ExcelWriter("../Data/parameters.xlsx") as writer:
    for file in files[1:]:
        f = file[:-4]
        df_csv = pd.read_csv(os.path.join(file_path, file), encoding='latin-1')
        df = pd.DataFrame({'Name': df_csv.columns.tolist()})  
        df.to_excel(writer, sheet_name=f, index=False)
        print("Finished extracting columns from file: " + f + ", N=" + str(i) + ", remaining: " + str(N-i))
        i+=1

<ipython-input-18-c165a4e6fe96>:4: UserWarning: Pandas requires version '1.4.3' or newer of 'xlsxwriter' (version '1.3.7' currently installed).
  df.to_excel("../Data/parameters.xlsx", sheet_name=f, index=False)
<ipython-input-18-c165a4e6fe96>:8: UserWarning: Pandas requires version '1.4.3' or newer of 'xlsxwriter' (version '1.3.7' currently installed).
  with pd.ExcelWriter("../Data/parameters.xlsx") as writer:


Finished extracting columns from file: patient_demo_registry, N=2, remaining: 25
Finished extracting columns from file: social_hx_tob, N=3, remaining: 24
Finished extracting columns from file: surgical_path_notes, N=4, remaining: 23
Finished extracting columns from file: enteredit_findings, N=5, remaining: 22
Finished extracting columns from file: social_hx_alc, N=6, remaining: 21


<ipython-input-18-c165a4e6fe96>:11: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  df_csv = pd.read_csv(os.path.join(file_path, file), encoding='latin-1')


Finished extracting columns from file: lab_results, N=7, remaining: 20
Finished extracting columns from file: problem_list, N=8, remaining: 19
Finished extracting columns from file: lab_sensitivity, N=9, remaining: 18
Finished extracting columns from file: pathology_findings, N=10, remaining: 17
Finished extracting columns from file: discharge_summary, N=11, remaining: 16
Finished extracting columns from file: procedures, N=12, remaining: 15
Finished extracting columns from file: encounter, N=13, remaining: 14
Finished extracting columns from file: recommendation_ie, N=14, remaining: 13


<ipython-input-18-c165a4e6fe96>:11: DtypeWarning: Columns (22) have mixed types. Specify dtype option on import or set low_memory=False.
  df_csv = pd.read_csv(os.path.join(file_path, file), encoding='latin-1')


Finished extracting columns from file: patient_data_ie, N=15, remaining: 12
Finished extracting columns from file: clinical_findings_ie, N=16, remaining: 11
Finished extracting columns from file: med_order, N=17, remaining: 10
Finished extracting columns from file: hormonal_mens, N=18, remaining: 9
Finished extracting columns from file: family_hx, N=19, remaining: 8
Finished extracting columns from file: risk_factors, N=20, remaining: 7


<ipython-input-18-c165a4e6fe96>:11: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df_csv = pd.read_csv(os.path.join(file_path, file), encoding='latin-1')


Finished extracting columns from file: procedure_notes, N=21, remaining: 6
Finished extracting columns from file: vitals, N=22, remaining: 5
Finished extracting columns from file: order_result, N=23, remaining: 4
Finished extracting columns from file: med_fill, N=24, remaining: 3
Finished extracting columns from file: diagnosis, N=25, remaining: 2
Finished extracting columns from file: img_pathology, N=26, remaining: 1
Finished extracting columns from file: img_procedures, N=27, remaining: 0
